In [1]:
import os

DATASET_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
for root, dirs, files in os.walk(DATASET_ROOT):
    depth = root.replace(DATASET_ROOT, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or root}/")
    for f in files:
        print(f"{indent}  {f}")

caai-revision-splits/
  credit_card/
    ood.parquet
    config.json
    id.parquet
    scaler.joblib
    id2.parquet
    train.parquet
  cic_ids2017/
    config.json
    scenario_A/
      ood.parquet
      id.parquet
      scaler.joblib
      id2.parquet
      train.parquet
    scenario_T/
      ood.parquet
      id.parquet
      scaler.joblib
      id2.parquet
      train.parquet


In [2]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATA_ROOT = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Credit card
cc_dir = os.path.join(DATA_ROOT, "credit_card")
cc_train = pd.read_parquet(os.path.join(cc_dir, "train.parquet"))
cc_id    = pd.read_parquet(os.path.join(cc_dir, "id.parquet"))
cc_id2   = pd.read_parquet(os.path.join(cc_dir, "id2.parquet"))
cc_ood   = pd.read_parquet(os.path.join(cc_dir, "ood.parquet"))
with open(os.path.join(cc_dir, "config.json")) as f:
    cc_config = json.load(f)
CC_FEATURES = cc_config["model_features"]
print(f"\nCredit card: Train={cc_train.shape}, ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")
print(f"Features ({len(CC_FEATURES)}):", CC_FEATURES)

# CIC-IDS2017, both scenarios
cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(DATA_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "train": pd.read_parquet(os.path.join(d, "train.parquet")),
        "id": pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2": pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood": pd.read_parquet(os.path.join(d, "ood.parquet")),
    }
with open(os.path.join(DATA_ROOT, "cic_ids2017", "config.json")) as f:
    cic_config = json.load(f)
CIC_FEATURES = cic_config["model_features"]
print(f"\nCIC-IDS2017 features ({len(CIC_FEATURES)}):", CIC_FEATURES[:5], "...")
for scen, d in cic_scenarios.items():
    print(f"{scen}: Train={d['train'].shape}, ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

Device: cuda
GPU: Tesla T4

Credit card: Train=(100774, 31), ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
Features (29): ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount']

CIC-IDS2017 features (67): ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets'] ...
scenario_T: Train=(910016, 82), ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: Train=(1190196, 82), ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)


In [3]:
def make_fit_val_split(train_df, val_frac=0.10, seed=SEED):
    shuffled = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = int(len(shuffled) * val_frac)
    val = shuffled.iloc[:n_val].copy()
    fit = shuffled.iloc[n_val:].copy()
    return fit, val

splits_for_training = {}

fit, val = make_fit_val_split(cc_train)
splits_for_training["credit_card"] = {"fit": fit, "val": val, "features": CC_FEATURES}
print(f"Credit card: fit={len(fit)}, val={len(val)}")

for scen, d in cic_scenarios.items():
    fit, val = make_fit_val_split(d["train"])
    splits_for_training[scen] = {"fit": fit, "val": val, "features": CIC_FEATURES}
    print(f"{scen}: fit={len(fit)}, val={len(val)}")

# Persist fit/val as parquet files so later notebooks (VAE, Deep SVDD) load the
# identical partition rather than re-shuffling with the same seed and hoping it matches
FITVAL_DIR = "/kaggle/working/fit_val_splits"
for name, d in splits_for_training.items():
    p = os.path.join(FITVAL_DIR, name)
    os.makedirs(p, exist_ok=True)
    d["fit"].to_parquet(os.path.join(p, "fit.parquet"), index=False)
    d["val"].to_parquet(os.path.join(p, "val.parquet"), index=False)
print(f"\nSaved fit/val splits to {FITVAL_DIR}")

Credit card: fit=90697, val=10077
scenario_T: fit=819015, val=91001
scenario_A: fit=1071177, val=119019

Saved fit/val splits to /kaggle/working/fit_val_splits


In [4]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def train_autoencoder(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True):
    input_dim = len(feature_cols)
    model = Autoencoder(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])
    loss_fn = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon = model(batch)
            loss = loss_fn(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_loss = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            val_recon = model(X_val)
            val_loss = loss_fn(val_recon, X_val).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"  Epoch {epoch:3d}  train_loss={train_loss:.6f}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - train_config["min_delta"]:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_loss, epoch


# Fixed training hyperparameters, logged for reproducibility (resolves reviewer comment D)
TRAIN_CONFIG = {
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "batch_size": 256,
    "max_epochs": 100,
    "patience": 10,
    "min_delta": 1e-5,
    "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
    "seed": SEED,
    "loss": "MSE",
}
print(json.dumps(TRAIN_CONFIG, indent=2))

{
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "batch_size": 256,
  "max_epochs": 100,
  "patience": 10,
  "min_delta": 1e-05,
  "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
  "seed": 42,
  "loss": "MSE"
}


In [5]:
ARCH_CONFIG_CC = {
    "hidden_dims": [20, 14],
    "latent_dim": 7,
    "architecture_rationale": "Shallow funnel (29->20->14->7->14->20->29); width chosen relative to the 29 input features, not tuned",
}
print(json.dumps(ARCH_CONFIG_CC, indent=2))

print("\nTraining Autoencoder — Credit Card")
ae_cc, history_cc, best_val_loss_cc, stopped_epoch_cc = train_autoencoder(
    fit_df=splits_for_training["credit_card"]["fit"],
    val_df=splits_for_training["credit_card"]["val"],
    feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_CC,
    train_config=TRAIN_CONFIG,
    device=DEVICE,
)
print(f"\nFinished at epoch {stopped_epoch_cc}, best val_loss={best_val_loss_cc:.6f}")

{
  "hidden_dims": [
    20,
    14
  ],
  "latent_dim": 7,
  "architecture_rationale": "Shallow funnel (29->20->14->7->14->20->29); width chosen relative to the 29 input features, not tuned"
}

Training Autoencoder — Credit Card
  Epoch   1  train_loss=0.852142  val_loss=0.741132
  Epoch   5  train_loss=0.521801  val_loss=0.506361
  Epoch  10  train_loss=0.430820  val_loss=0.425912
  Epoch  15  train_loss=0.399459  val_loss=0.395714
  Epoch  20  train_loss=0.371702  val_loss=0.374207
  Epoch  25  train_loss=0.353190  val_loss=0.349028
  Epoch  30  train_loss=0.344767  val_loss=0.347146
  Epoch  35  train_loss=0.338277  val_loss=0.334833
  Epoch  40  train_loss=0.332991  val_loss=0.330944
  Epoch  45  train_loss=0.329156  val_loss=0.336115
  Epoch  50  train_loss=0.325243  val_loss=0.320391
  Epoch  55  train_loss=0.322841  val_loss=0.328158
  Epoch  60  train_loss=0.318134  val_loss=0.314099
  Epoch  65  train_loss=0.315396  val_loss=0.309100
  Epoch  70  train_loss=0.313521  val_loss

In [6]:
TRAIN_CONFIG["max_epochs"] = 300
print("Updated TRAIN_CONFIG:", json.dumps(TRAIN_CONFIG, indent=2))

print("\nTraining Autoencoder — Credit Card (re-run)")
ae_cc, history_cc, best_val_loss_cc, stopped_epoch_cc = train_autoencoder(
    fit_df=splits_for_training["credit_card"]["fit"],
    val_df=splits_for_training["credit_card"]["val"],
    feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_CC,
    train_config=TRAIN_CONFIG,
    device=DEVICE,
)
print(f"\nFinished at epoch {stopped_epoch_cc}, best val_loss={best_val_loss_cc:.6f}")

Updated TRAIN_CONFIG: {
  "optimizer": "Adam",
  "learning_rate": 0.001,
  "batch_size": 256,
  "max_epochs": 300,
  "patience": 10,
  "min_delta": 1e-05,
  "early_stopping_criterion": "validation MSE, no improvement > min_delta for `patience` consecutive epochs",
  "seed": 42,
  "loss": "MSE"
}

Training Autoencoder — Credit Card (re-run)
  Epoch   1  train_loss=0.882202  val_loss=0.750595
  Epoch   5  train_loss=0.526123  val_loss=0.522717
  Epoch  10  train_loss=0.453811  val_loss=0.458163
  Epoch  15  train_loss=0.421860  val_loss=0.424864
  Epoch  20  train_loss=0.401376  val_loss=0.409591
  Epoch  25  train_loss=0.385087  val_loss=0.388171
  Epoch  30  train_loss=0.372727  val_loss=0.372587
  Epoch  35  train_loss=0.364360  val_loss=0.364958
  Epoch  40  train_loss=0.357948  val_loss=0.357055
  Epoch  45  train_loss=0.352108  val_loss=0.357429
  Epoch  50  train_loss=0.348123  val_loss=0.346150
  Epoch  55  train_loss=0.344506  val_loss=0.343185
  Epoch  60  train_loss=0.341462  

In [7]:
def compute_reconstruction_error(model, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        recon = model(X)
        errors = ((recon - X) ** 2).mean(dim=1).cpu().numpy()
    return errors

def score_split(model, df, feature_cols, device, split_name):
    errors = compute_reconstruction_error(model, df, feature_cols, device)
    out = pd.DataFrame({
        "row_id": np.arange(len(df)),
        "Class": df["Class"].values,
        "reconstruction_error": errors,
    })
    print(f"{split_name}: benign mean={out[out.Class==0].reconstruction_error.mean():.4f}, "
          f"anomaly mean={out[out.Class==1].reconstruction_error.mean():.4f} "
          f"(n_benign={sum(out.Class==0)}, n_anomaly={sum(out.Class==1)})")
    return out

print("=== AE — Credit Card reconstruction error by split/class ===")
ae_cc_scores = {
    "id":  score_split(ae_cc, cc_id,  CC_FEATURES, DEVICE, "ID"),
    "id2": score_split(ae_cc, cc_id2, CC_FEATURES, DEVICE, "ID2"),
    "ood": score_split(ae_cc, cc_ood, CC_FEATURES, DEVICE, "OOD"),
}

=== AE — Credit Card reconstruction error by split/class ===
ID: benign mean=0.3029, anomaly mean=25.7962 (n_benign=21595, n_anomaly=136)
ID2: benign mean=0.2964, anomaly mean=21.4928 (n_benign=21595, n_anomaly=136)
OOD: benign mean=0.6406, anomaly mean=18.0341 (n_benign=139289, n_anomaly=201)


In [8]:
AE_DIR = os.path.join(OUTPUT_DIR, "ae_credit_card")
os.makedirs(AE_DIR, exist_ok=True)

torch.save(ae_cc.state_dict(), os.path.join(AE_DIR, "model.pt"))

ae_cc_config = {
    "model": "Autoencoder",
    "dataset": "credit_card",
    "architecture": ARCH_CONFIG_CC,
    "training": TRAIN_CONFIG,
    "stopped_epoch": stopped_epoch_cc,
    "best_val_loss": best_val_loss_cc,
    "fit_size": len(splits_for_training["credit_card"]["fit"]),
    "val_size": len(splits_for_training["credit_card"]["val"]),
}
with open(os.path.join(AE_DIR, "config.json"), "w") as f:
    json.dump(ae_cc_config, f, indent=2)

for split_name, scores_df in ae_cc_scores.items():
    scores_df.to_parquet(os.path.join(AE_DIR, f"scores_{split_name}.parquet"), index=False)

print("Saved to:", AE_DIR)
print(os.listdir(AE_DIR))

Saved to: /kaggle/working/models/ae_credit_card
['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']


In [9]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc_layers = []
        prev = input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.encoder_body = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)

        dec_layers = []
        prev = latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        self.decoder_body = nn.Sequential(*dec_layers)
        self.fc_out_mu = nn.Linear(prev, input_dim)
        self.fc_out_logvar = nn.Linear(prev, input_dim)

    def encode(self, x):
        h = self.encoder_body(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.decoder_body(z)
        out_mu = self.fc_out_mu(h)
        out_logvar = torch.clamp(self.fc_out_logvar(h), min=-10, max=10)
        return out_mu, out_logvar

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        out_mu, out_logvar = self.decode(z)
        return out_mu, out_logvar, mu_z, logvar_z


def vae_elbo_loss(x, out_mu, out_logvar, mu_z, logvar_z):
    recon_nll = 0.5 * (out_logvar + ((x - out_mu) ** 2) / torch.exp(out_logvar) + np.log(2 * np.pi))
    recon_nll = recon_nll.sum(dim=1).mean()
    kl = -0.5 * torch.sum(1 + logvar_z - mu_z.pow(2) - logvar_z.exp(), dim=1).mean()
    return recon_nll + kl, recon_nll, kl


def train_vae(fit_df, val_df, feature_cols, arch_config, train_config, device, verbose=True):
    input_dim = len(feature_cols)
    model = VAE(input_dim, arch_config["hidden_dims"], arch_config["latent_dim"]).to(device)

    X_fit = torch.tensor(fit_df[feature_cols].values, dtype=torch.float32)
    X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32).to(device)

    fit_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_fit),
        batch_size=train_config["batch_size"], shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=train_config["learning_rate"])

    best_val_elbo = float("inf")
    best_state = None
    epochs_no_improve = 0
    history = {"train_elbo": [], "val_elbo": [], "val_recon_nll": [], "val_kl": []}

    for epoch in range(1, train_config["max_epochs"] + 1):
        model.train()
        epoch_losses = []
        for (batch,) in fit_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out_mu, out_logvar, mu_z, logvar_z = model(batch)
            loss, _, _ = vae_elbo_loss(batch, out_mu, out_logvar, mu_z, logvar_z)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        train_elbo = float(np.mean(epoch_losses))

        model.eval()
        with torch.no_grad():
            out_mu, out_logvar, mu_z, logvar_z = model(X_val)
            val_elbo, val_recon_nll, val_kl = vae_elbo_loss(X_val, out_mu, out_logvar, mu_z, logvar_z)
            val_elbo, val_recon_nll, val_kl = val_elbo.item(), val_recon_nll.item(), val_kl.item()

        history["train_elbo"].append(train_elbo)
        history["val_elbo"].append(val_elbo)
        history["val_recon_nll"].append(val_recon_nll)
        history["val_kl"].append(val_kl)

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"  Epoch {epoch:3d}  train_elbo={train_elbo:.4f}  val_elbo={val_elbo:.4f}  "
                  f"(recon_nll={val_recon_nll:.4f}, kl={val_kl:.4f})")

        if val_elbo < best_val_elbo - train_config["min_delta"]:
            best_val_elbo = val_elbo
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= train_config["patience"]:
                if verbose:
                    print(f"  Early stopping at epoch {epoch} (no improvement for {train_config['patience']} epochs)")
                break

    model.load_state_dict(best_state)
    return model, history, best_val_elbo, epoch

In [10]:
ARCH_CONFIG_CC_VAE = {
    "hidden_dims": [20, 14],
    "latent_dim": 7,
    "architecture_rationale": "Mirrors AE-Credit-Card exactly (29->20->14->7->14->20->29) for a fair comparison; only the VAE-specific heads (mu/logvar) differ",
}
print(json.dumps(ARCH_CONFIG_CC_VAE, indent=2))

print("\nTraining VAE — Credit Card")
vae_cc, history_vae_cc, best_val_elbo_cc, stopped_epoch_vae_cc = train_vae(
    fit_df=splits_for_training["credit_card"]["fit"],
    val_df=splits_for_training["credit_card"]["val"],
    feature_cols=CC_FEATURES,
    arch_config=ARCH_CONFIG_CC_VAE,
    train_config=TRAIN_CONFIG,
    device=DEVICE,
)
print(f"\nFinished at epoch {stopped_epoch_vae_cc}, best val_elbo={best_val_elbo_cc:.4f}")

{
  "hidden_dims": [
    20,
    14
  ],
  "latent_dim": 7,
  "architecture_rationale": "Mirrors AE-Credit-Card exactly (29->20->14->7->14->20->29) for a fair comparison; only the VAE-specific heads (mu/logvar) differ"
}

Training VAE — Credit Card
  Epoch   1  train_elbo=39.9089  val_elbo=35.9607  (recon_nll=34.6119, kl=1.3487)
  Epoch   5  train_elbo=30.3803  val_elbo=30.3205  (recon_nll=28.1726, kl=2.1479)
  Epoch  10  train_elbo=27.4388  val_elbo=27.7263  (recon_nll=24.3078, kl=3.4186)
  Epoch  15  train_elbo=26.4558  val_elbo=26.9107  (recon_nll=23.2082, kl=3.7025)
  Epoch  20  train_elbo=26.0276  val_elbo=26.4354  (recon_nll=22.6214, kl=3.8140)
  Epoch  25  train_elbo=25.7049  val_elbo=26.2106  (recon_nll=22.2669, kl=3.9437)
  Epoch  30  train_elbo=25.5082  val_elbo=26.0263  (recon_nll=21.9547, kl=4.0716)
  Epoch  35  train_elbo=25.2928  val_elbo=25.7990  (recon_nll=21.6982, kl=4.1008)
  Epoch  40  train_elbo=25.1397  val_elbo=25.6077  (recon_nll=21.4522, kl=4.1555)
  Epoch  45  

In [11]:
def compute_vae_nll(model, df, feature_cols, device, n_samples=10):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        mu_z, logvar_z = model.encode(X)
        nlls = []
        for _ in range(n_samples):
            z = model.reparameterize(mu_z, logvar_z)
            out_mu, out_logvar = model.decode(z)
            nll = 0.5 * (out_logvar + ((X - out_mu) ** 2) / torch.exp(out_logvar) + np.log(2 * np.pi))
            nlls.append(nll.sum(dim=1))
        avg_nll = torch.stack(nlls).mean(dim=0).cpu().numpy()
    return avg_nll

def score_split_vae(model, df, feature_cols, device, split_name, n_samples=10):
    scores = compute_vae_nll(model, df, feature_cols, device, n_samples)
    out = pd.DataFrame({
        "row_id": np.arange(len(df)),
        "Class": df["Class"].values,
        "vae_nll": scores,
    })
    print(f"{split_name}: benign mean={out[out.Class==0].vae_nll.mean():.4f}, "
          f"anomaly mean={out[out.Class==1].vae_nll.mean():.4f} "
          f"(n_benign={sum(out.Class==0)}, n_anomaly={sum(out.Class==1)})")
    return out

print("=== VAE — Credit Card negative log-likelihood by split/class ===")
vae_cc_scores = {
    "id":  score_split_vae(vae_cc, cc_id,  CC_FEATURES, DEVICE, "ID"),
    "id2": score_split_vae(vae_cc, cc_id2, CC_FEATURES, DEVICE, "ID2"),
    "ood": score_split_vae(vae_cc, cc_ood, CC_FEATURES, DEVICE, "OOD"),
}

=== VAE — Credit Card negative log-likelihood by split/class ===
ID: benign mean=19.2076, anomaly mean=82.1883 (n_benign=21595, n_anomaly=136)
ID2: benign mean=19.2576, anomaly mean=81.0238 (n_benign=21595, n_anomaly=136)
OOD: benign mean=71.9296, anomaly mean=85.0422 (n_benign=139289, n_anomaly=201)


In [12]:
def diagnose_output_variance(model, df, feature_cols, device):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        mu_z, logvar_z = model.encode(X)
        z = mu_z  # use mean, no sampling noise for this diagnostic
        _, out_logvar = model.decode(z)
    out_logvar = out_logvar.cpu().numpy()
    pct_at_clamp = (out_logvar <= -9.9).mean() * 100
    return out_logvar.mean(), out_logvar.min(), out_logvar.max(), pct_at_clamp

for split_name, split_df in [("ID", cc_id), ("ID2", cc_id2), ("OOD", cc_ood)]:
    mean_lv, min_lv, max_lv, pct_clamped = diagnose_output_variance(vae_cc, split_df, CC_FEATURES, DEVICE)
    print(f"{split_name}: mean out_logvar={mean_lv:.3f}, min={min_lv:.3f}, max={max_lv:.3f}, "
          f"% values at clamp (-9.9)={pct_clamped:.2f}%")

ID: mean out_logvar=-1.493, min=-9.853, max=10.000, % values at clamp (-9.9)=0.00%
ID2: mean out_logvar=-1.492, min=-9.866, max=10.000, % values at clamp (-9.9)=0.00%
OOD: mean out_logvar=-1.224, min=-8.639, max=10.000, % values at clamp (-9.9)=0.00%


In [13]:
VAE_DIR = os.path.join(OUTPUT_DIR, "vae_credit_card")
os.makedirs(VAE_DIR, exist_ok=True)

torch.save(vae_cc.state_dict(), os.path.join(VAE_DIR, "model.pt"))

vae_cc_config = {
    "model": "VAE",
    "dataset": "credit_card",
    "architecture": ARCH_CONFIG_CC_VAE,
    "training": TRAIN_CONFIG,
    "stopped_epoch": stopped_epoch_vae_cc,
    "best_val_elbo": best_val_elbo_cc,
    "scoring_method": "negative log-likelihood, Monte Carlo averaged over 10 latent samples",
    "fit_size": len(splits_for_training["credit_card"]["fit"]),
    "val_size": len(splits_for_training["credit_card"]["val"]),
    "output_variance_diagnostic": {
        "id_mean_logvar": -1.493, "id2_mean_logvar": -1.492, "ood_mean_logvar": -1.224,
        "pct_at_clamp": 0.0,
        "conclusion": "No variance collapse; OOD NLL increase reflects genuine reconstruction degradation",
    },
}
with open(os.path.join(VAE_DIR, "config.json"), "w") as f:
    json.dump(vae_cc_config, f, indent=2)

for split_name, scores_df in vae_cc_scores.items():
    scores_df.to_parquet(os.path.join(VAE_DIR, f"scores_{split_name}.parquet"), index=False)

print("Saved to:", VAE_DIR)
print(os.listdir(VAE_DIR))

Saved to: /kaggle/working/models/vae_credit_card
['scores_id2.parquet', 'scores_ood.parquet', 'config.json', 'model.pt', 'scores_id.parquet']


In [14]:
import shutil

shutil.make_archive("/kaggle/working/models_archive", "zip", "/kaggle/working/models")
print("Created: /kaggle/working/models_archive.zip")
print("Size (MB):", os.path.getsize("/kaggle/working/models_archive.zip") / (1024*1024))

Created: /kaggle/working/models_archive.zip
Size (MB): 2.586578369140625
